In [ ]:
# ============================================================
# GOOGLE COLAB
# VEHICLE + PEDESTRIAN DETECTION IN A 20-SECOND ROAD VIDEO
#
# Detects:
# Person
# Bicycle
# Sedan
# SUV
# Motorcycle / Two Wheeler
# Bus
# Truck
#
# Uses:
# YOLO11 + OpenCV
# ============================================================

!pip install -q ultralytics opencv-python

from ultralytics import YOLO
import cv2
from google.colab import files
from IPython.display import HTML, display
from base64 import b64encode
import os


In [ ]:
# ============================================================
# 1. UPLOAD YOUR ROAD / HIGHWAY VIDEO
# ============================================================

print("Upload your road/highway video:")
uploaded = files.upload()

input_video = list(uploaded.keys())[0]

print("Input video:", input_video)


In [ ]:
# ============================================================
# 2. LOAD YOLO MODEL
# ============================================================

model = YOLO("yolo11n.pt")


# ============================================================
# 3. OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(input_video)

if not cap.isOpened():
    raise Exception("Could not open the video.")


width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

print("Width :", width)
print("Height:", height)
print("FPS   :", fps)


# ============================================================
# 4. LIMIT VIDEO TO 20 SECONDS
# ============================================================

max_frames = int(fps * 20)


In [ ]:
# ============================================================
# 5. OUTPUT VIDEO
# ============================================================

output_video = "road_detection_output.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    output_video,
    fourcc,
    fps,
    (width, height)
)


# ============================================================
# 6. YOLO COCO CLASS IDS
# ============================================================
#
# 0 = person
# 1 = bicycle
# 2 = car
# 3 = motorcycle
# 5 = bus
# 7 = truck
#
# ============================================================

classes_to_detect = [0, 1, 2, 3, 5, 7]


# ============================================================
# 7. COLORS FOR EACH OBJECT TYPE
# ============================================================

colors = {

    "Pedestrian": (0, 255, 255),

    "Bicycle": (255, 255, 0),

    "Sedan": (0, 255, 0),

    "SUV": (255, 0, 255),

    "Two Wheeler": (0, 165, 255),

    "Bus": (255, 0, 0),

    "Truck": (0, 0, 255)
}


In [ ]:


# ============================================================
# 8. PROCESS VIDEO FRAME BY FRAME
# ============================================================

frame_number = 0


while True:

    ret, frame = cap.read()

    if not ret:
        break


    frame_number += 1


    # Stop after 20 seconds

    if frame_number > max_frames:
        break


    # ========================================================
    # YOLO OBJECT DETECTION
    # ========================================================

    results = model(
        frame,
        classes=classes_to_detect,
        conf=0.35,
        verbose=False
    )


    result = results[0]


    # ========================================================
    # PROCESS EACH DETECTED OBJECT
    # ========================================================

    for box in result.boxes:


        # ----------------------------------------------------
        # Bounding box coordinates
        # ----------------------------------------------------

        x1, y1, x2, y2 = map(
            int,
            box.xyxy[0].tolist()
        )


        # ----------------------------------------------------
        # Confidence
        # ----------------------------------------------------

        confidence = float(box.conf[0])


        # ----------------------------------------------------
        # YOLO class
        # ----------------------------------------------------

        class_id = int(box.cls[0])


        # Bounding box dimensions

        box_width = x2 - x1
        box_height = y2 - y1

        box_area = box_width * box_height


        # ====================================================
        # 9. ASSIGN VEHICLE NAME
        # ====================================================


        # PERSON
        if class_id == 0:

            object_name = "Pedestrian"


        # BICYCLE
        elif class_id == 1:

            object_name = "Bicycle"


        # CAR
        elif class_id == 2:

            # -----------------------------------------------
            # Demonstration heuristic:
            #
            # small bounding box -> Sedan
            # large bounding box -> SUV
            #
            # This threshold may need adjustment depending
            # on your video resolution and camera position.
            # -----------------------------------------------

            if box_area > 50000:

                object_name = "SUV"

            else:

                object_name = "Sedan"


        # MOTORCYCLE
        elif class_id == 3:

            object_name = "Two Wheeler"


        # BUS
        elif class_id == 5:

            object_name = "Bus"


        # TRUCK
        elif class_id == 7:

            object_name = "Truck"


        else:

            continue


        # ====================================================
        # 10. GET OBJECT COLOR
        # ====================================================

        color = colors[object_name]


        # ====================================================
        # 11. DRAW BOUNDING BOX
        # ====================================================

        cv2.rectangle(
            frame,
            (x1, y1),
            (x2, y2),
            color,
            3
        )


        # ====================================================
        # 12. LABEL
        # ====================================================

        label = f"{object_name} {confidence:.2f}"


        # Background behind label

        (text_width, text_height), _ = cv2.getTextSize(
            label,
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            2
        )


        cv2.rectangle(
            frame,
            (x1, y1 - text_height - 12),
            (x1 + text_width + 10, y1),
            color,
            -1
        )


        # Label text

        cv2.putText(
            frame,
            label,
            (x1 + 5, y1 - 7),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2
        )


    # ========================================================
    # 13. DISPLAY VIDEO INFORMATION
    # ========================================================

    time_seconds = frame_number / fps


    cv2.putText(
        frame,
        f"Time: {time_seconds:.1f} sec",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )


    writer.write(frame)


# ============================================================
# 14. CLEANUP
# ============================================================

cap.release()
writer.release()


print("Detection completed.")
print("Output:", output_video)


# ============================================================
# 15. CONVERT VIDEO FOR GOOGLE COLAB DISPLAY
# ============================================================

display_video = "road_detection_display.mp4"


os.system(
    f"ffmpeg -y -loglevel error "
    f"-i {output_video} "
    f"-vcodec libx264 "
    f"{display_video}"
)


In [ ]:
# ============================================================
# 16. DISPLAY RESULT INSIDE COLAB
# ============================================================

mp4 = open(display_video, "rb").read()

video_data = (
    "data:video/mp4;base64,"
    + b64encode(mp4).decode()
)


display(

    HTML(

        f'''

        <video width="900"
               controls
               autoplay
               loop>

            <source
                src="{video_data}"
                type="video/mp4">

        </video>

        '''
    )
)